# Tối ưu hóa có ràng buộc — Genetic Algorithm vs SciPy SLSQP

Notebook chứa toàn bộ thuật toán (parser, Genetic Algorithm, SciPy SLSQP) — không phụ thuộc file `.py` bên ngoài nào.

**Cách dùng:** chạy các cell từ trên xuống (`Run All`). Muốn đổi bài toán thì
sửa form ở cell ①, bấm **Áp dụng bài toán**, rồi chạy lại từ cell ②.

In [6]:
import functools
import re
import time

import numpy as np
import scipy
import sympy
sp = sympy

from dataclasses import dataclass
from scipy.optimize import minimize
from sympy.parsing.sympy_parser import (
    parse_expr,
    standard_transformations,
    implicit_multiplication,
    convert_xor,
    function_exponentiation,
)


# ============================================================
# 1. PARSER
# ============================================================

# LƯU Ý QUAN TRỌNG:
#
# KHÔNG dùng implicit_multiplication_application ở đây, vì nó bao
# gồm split_symbols - transformation này cắt tên biến nhiều ký tự
# thành tích các chữ cái đơn:
#
#       x1^2 + x2^2   ->   x*1**2 + x*2**2   ->   5*x
#       cost          ->   c*o*s*t
#
# tức là chương trình sẽ âm thầm giải một bài toán khác hẳn.
#
# Dùng implicit_multiplication (không có split_symbols) thì
# "2x + 3y" vẫn hiểu được, còn "x1", "x2" giữ nguyên là biến.

TRANSFORMATIONS = standard_transformations + (
    implicit_multiplication,
    convert_xor,
    function_exponentiation,
)

# Các hàm / hằng số toán học mà user được phép nhập.
# Đây cũng là các tên KHÔNG được dùng làm tên biến.
LOCAL_DICT = {
    "sin": sp.sin,
    "cos": sp.cos,
    "tan": sp.tan,
    "asin": sp.asin,
    "acos": sp.acos,
    "atan": sp.atan,
    "sinh": sp.sinh,
    "cosh": sp.cosh,
    "tanh": sp.tanh,
    "exp": sp.exp,
    "log": sp.log,
    "ln": sp.log,
    "sqrt": sp.sqrt,
    "abs": sp.Abs,
    "Abs": sp.Abs,
    "pi": sp.pi,
    "e": sp.E,
    "E": sp.E,
}

# Giới hạn namespace mà parse_expr nhìn thấy.
#
# Mặc định parse_expr dùng toàn bộ namespace của sympy, nên những
# tên biến hoàn toàn hợp lệ như N, O, Q, S, I, beta, gamma... bị
# hiểu thành đối tượng sympy thay vì biến (I -> đơn vị ảo,
# N -> hàm evalf, beta -> hàm beta, ...).
#
# Chỉ để lại đúng những gì bộ parse cần để dựng biểu thức; mọi
# tên khác sẽ tự động trở thành Symbol.
GLOBAL_DICT = {
    "Symbol": sp.Symbol,
    "Integer": sp.Integer,
    "Float": sp.Float,
    "Rational": sp.Rational,
}

RESERVED_NAMES = set(LOCAL_DICT)

# Ký hiệu so sánh, xếp để bắt "<=" trước "<"
COMPARISON_PATTERN = re.compile(r"<=|>=|==|<|>|=")


@dataclass
class ParsedConstraint:
    # ineq: expr >= 0
    # eq:   expr = 0
    kind: str
    expr: sp.Expr
    original: str


def parse_math_expr(text):
    """
    Parse biểu thức toán học tự nhiên.

    Ví dụ:
        x^2 + y^2
        2x + 3y
        x1^2 + x2^2        (tên biến nhiều ký tự được giữ nguyên)
        sin(x) + cos(y)
        e^x + log(y)
    """

    text = text.strip()

    if not text:
        raise ValueError("Biểu thức rỗng.")

    try:
        return parse_expr(
            text,
            local_dict=LOCAL_DICT,
            global_dict=GLOBAL_DICT,
            transformations=TRANSFORMATIONS,
            evaluate=True,
        )

    except Exception as error:
        raise ValueError(
            f"Không đọc được biểu thức {text!r}: {error}"
        ) from error


def parse_constraint(text):
    """
    Chuẩn hóa constraint về:

        g(x) >= 0       nếu inequality
        h(x) = 0        nếu equality

    Ví dụ:
        x^2 + y^2 <= 9
    thành:
        9 - x^2 - y^2 >= 0
    """

    text = text.strip()

    operators = COMPARISON_PATTERN.findall(text)

    if not operators:
        raise ValueError(
            f"Ràng buộc {text!r} phải chứa <=, >= hoặc =."
        )

    # Chuỗi kép "0 <= x <= 5" do parse_constraints() tách trước khi
    # gọi vào đây, nên tới đây mà còn nhiều dấu so sánh là thật sự sai.
    if len(operators) > 1:
        raise ValueError(
            f"Ràng buộc {text!r} có quá nhiều dấu so sánh."
        )

    operator = operators[0]

    if operator in ("<", ">"):
        raise ValueError(
            f"Ràng buộc {text!r} dùng bất đẳng thức nghiêm ngặt. "
            "Tối ưu số cần miền đóng, hãy dùng "
            f"'{operator}=' thay cho '{operator}'."
        )

    lhs_text, rhs_text = text.split(operator, 1)

    lhs = parse_math_expr(lhs_text)
    rhs = parse_math_expr(rhs_text)

    if operator == "<=":
        # lhs <= rhs
        # rhs - lhs >= 0
        expr = rhs - lhs
        kind = "ineq"

    elif operator == ">=":
        # lhs >= rhs
        # lhs - rhs >= 0
        expr = lhs - rhs
        kind = "ineq"

    else:
        # lhs = rhs
        # lhs - rhs = 0
        expr = lhs - rhs
        kind = "eq"

    # expand() đủ để gom hạng tử và rẻ hơn simplify() rất nhiều
    return ParsedConstraint(
        kind=kind,
        expr=sp.expand(expr),
        original=text,
    )


def parse_constraints(text):
    """
    Đọc MỘT dòng ràng buộc, trả về danh sách ràng buộc đã chuẩn hóa.

    Bình thường một dòng cho một ràng buộc. Riêng dạng chuỗi kép thì
    tách làm hai:

        0 <= x <= 2     ->     0 <= x     và     x <= 2
    """

    text = text.strip()

    operators = COMPARISON_PATTERN.findall(text)

    if len(operators) <= 1:
        return [parse_constraint(text)]

    if len(operators) > 2:
        raise ValueError(
            f"Ràng buộc {text!r} có nhiều hơn hai dấu so sánh."
        )

    operator = operators[0]

    if operators[1] != operator or operator not in ("<=", ">="):
        raise ValueError(
            f"Ràng buộc {text!r} có hai dấu so sánh không cùng chiều. "
            "Dạng chuỗi kép chỉ nhận 'a <= x <= b' hoặc 'a >= x >= b'."
        )

    # Cắt tại đúng hai vị trí toán tử
    dau = text.index(operator)
    sau = text.index(operator, dau + len(operator))

    trai = text[:dau]
    giua = text[dau + len(operator):sau]
    phai = text[sau + len(operator):]

    return [
        parse_constraint(f"{trai}{operator}{giua}"),
        parse_constraint(f"{giua}{operator}{phai}"),
    ]


# ============================================================
# 2. BUILD OPTIMIZATION PROBLEM
# ============================================================

def _natural_sort_key(symbol):
    """
    Sắp biến theo thứ tự tự nhiên: x1, x2, x10
    thay vì thứ tự chữ cái: x1, x10, x2
    """

    parts = re.split(r"(\d+)", symbol.name)

    return [
        (1, int(part), "") if part.isdigit() else (0, 0, part)
        for part in parts
    ]


def _implicit_products(symbols):
    """
    Suy ra phép nhân ngầm từ tên biến bị dính liền.

    Bỏ split_symbols của sympy là cần thiết để 'x1', 'x2', 'cost' giữ
    nguyên là biến. Cái giá là 'xy' cũng thành một biến, trong khi
    người dùng gõ '2xy' gần như luôn có ý là 2*x*y.

    Quy tắc tách, chỉ dựa vào bằng chứng trong CHÍNH bài toán: một tên
    nhiều chữ cái được tách thành tích khi MỌI chữ cái của nó đều đã
    là biến ở chỗ khác.

        4x^2 - 2xy + 6y^2   ->  có x, có y  ->  xy tách thành x*y
        cost + x            ->  c,o,s,t không phải biến  ->  giữ 'cost'
        x1^2 + x2^2         ->  có chữ số  ->  không đụng tới
        min xy              ->  không có x, y nào khác  ->  giữ 'xy'

    Trả về dict thay thế cho Expr.subs(), rỗng nếu không có gì để tách.
    """

    don_le = {
        symbol.name
        for symbol in symbols
        if len(symbol.name) == 1 and symbol.name.isalpha()
    }

    thay_the = {}

    for symbol in symbols:
        ten = symbol.name

        if len(ten) < 2 or not ten.isalpha():
            continue

        if all(chu in don_le for chu in ten):
            tich = sp.Integer(1)
            for chu in ten:
                tich *= sp.Symbol(chu)
            thay_the[symbol] = tich

    return thay_the


def build_problem(objective_text, constraint_texts):

    # Parse objective
    objective_expr = parse_math_expr(objective_text)

    # Parse constraints (một dòng có thể sinh ra hai ràng buộc)
    constraints = [
        parsed
        for text in constraint_texts
        for parsed in parse_constraints(text)
    ]

    # --------------------------------------------------------
    # Tự động tìm tất cả biến
    # --------------------------------------------------------

    symbols = set(objective_expr.free_symbols)

    for constraint in constraints:
        symbols |= constraint.expr.free_symbols

    # Tách tên dính liền thành phép nhân: '2xy' -> 2*x*y
    thay_the = _implicit_products(symbols)

    if thay_the:
        objective_expr = sp.expand(objective_expr.subs(thay_the))

        constraints = [
            ParsedConstraint(
                kind=c.kind,
                expr=sp.expand(c.expr.subs(thay_the)),
                original=c.original,
            )
            for c in constraints
        ]

        symbols = set(objective_expr.free_symbols)

        for constraint in constraints:
            symbols |= constraint.expr.free_symbols

    variables = sorted(symbols, key=_natural_sort_key)

    if not variables:
        raise ValueError("Không tìm thấy biến quyết định.")

    # --------------------------------------------------------
    # Symbolic -> numerical
    # --------------------------------------------------------

    objective_raw = sp.lambdify(
        variables,
        objective_expr,
        modules="numpy"
    )

    constraint_raw = [
        sp.lambdify(
            variables,
            c.expr,
            modules="numpy"
        )
        for c in constraints
    ]

    # Gradient ky hieu cua tung rang buoc, dung de chuan hoa thang do
    # vi pham (xem build_scaled_constraint_value)
    gradient_raw = [
        sp.lambdify(
            variables,
            [sp.diff(c.expr, v) for v in variables],
            modules="numpy"
        )
        for c in constraints
    ]

    def objective(x):
        try:
            value = float(
                np.asarray(objective_raw(*x)).reshape(())
            )

            if np.isfinite(value):
                return value

        except Exception:
            pass

        return np.inf

    def constraint_value(index, x):
        try:
            value = float(
                np.asarray(
                    constraint_raw[index](*x)
                ).reshape(())
            )

            if np.isfinite(value):
                return value

        except Exception:
            pass

        return np.nan

    def constraint_gradient(index, x):
        try:
            gradient = np.asarray(
                gradient_raw[index](*x),
                dtype=float
            ).ravel()

            if np.all(np.isfinite(gradient)):
                return gradient

        except Exception:
            pass

        return np.full(len(variables), np.nan)

    return (
        objective_expr,
        constraints,
        variables,
        objective,
        constraint_value,
        constraint_gradient,
    )


# ============================================================
# 3. CONSTRAINT HANDLING
# ============================================================

# Mức phạt cho điểm nằm ngoài miền xác định (log(âm), chia 0, ...)
OUT_OF_DOMAIN_VIOLATION = 1e6

FEASIBILITY_TOLERANCE = 1e-6

# Hộp dùng để SINH quần thể ban đầu và đặt thang bước đột biến
# (sigma = mutation_scale x độ rộng hộp). KHÔNG phải một cái lồng:
# GA không bị cắt về hộp nên có thể di cư ra ngoài nếu nghiệm nằm
# ngoài. Muốn chặn thật thì viết thành ràng buộc, ví dụ 'x >= 0'.
INIT_BOX = (-5.0, 5.0)


def build_scaled_constraint_value(
    constraints,
    constraint_value,
    constraint_gradient,
    init_box,
    n_samples=256,
    seed=0,
):
    """
    Chuẩn hóa mỗi ràng buộc theo độ lớn gradient điển hình của nó.

    Vì sao cần: mức vi phạm |h(x)| phụ thuộc vào việc người dùng
    viết ràng buộc thế nào, trong khi tolerance lại là một hằng số
    tuyệt đối. Hai cách viết TƯƠNG ĐƯƠNG cho kết quả khác nhau:

        x + y = 1                  ->  tolerance hiệu dụng 1e-6
        0.001x + 0.001y = 0.001    ->  tolerance hiệu dụng 1e-3

    Cách viết thứ hai từng cho f = 0.49925, tức là THẤP HƠN cực
    tiểu thật 0.5 - một giá trị bất khả thi về mặt toán học, vì
    chương trình coi là khả thi những điểm thực ra còn vi phạm.

    Chia g(x) cho ||∇g|| điển hình làm mức vi phạm trở thành xấp xỉ
    KHOẢNG CÁCH tới mặt ràng buộc: nhân cả ràng buộc với hằng số k
    thì tử số và mẫu số cùng nhân k, kết quả không đổi.

    Dùng một hệ số HẰNG cho mỗi ràng buộc (trung vị của ||∇g|| trên
    một mẫu ngẫu nhiên trong miền) thay vì ||∇g(x)|| tại từng điểm:
    hệ số hằng vẫn đạt bất biến tỉ lệ, đồng thời tránh trường hợp
    gradient suy biến (∇g = 0, ví dụ x^2+y^2-1 tại gốc tọa độ) làm
    mẫu số bằng 0.

    Hệ số dương nên phép chia không đổi dấu, không đổi nghiệm - với
    SLSQP đây còn là bước cải thiện điều kiện số (constraint
    scaling) được khuyến nghị trong tối ưu phi tuyến.
    """

    if not constraints:
        return constraint_value

    init_box = np.asarray(init_box, dtype=float)

    rng = np.random.default_rng(seed)

    sample = rng.uniform(
        init_box[:, 0],
        init_box[:, 1],
        size=(n_samples, len(init_box))
    )

    scales = np.ones(len(constraints))

    for i in range(len(constraints)):

        norms = [
            norm
            for norm in (
                float(
                    np.linalg.norm(constraint_gradient(i, x))
                )
                for x in sample
            )
            if np.isfinite(norm) and norm > 0
        ]

        if norms:
            scales[i] = float(np.median(norms))

    def scaled_constraint_value(index, x):
        return constraint_value(index, x) / scales[index]

    scaled_constraint_value.scales = scales

    return scaled_constraint_value


def constraint_violations(x, constraints, constraint_value):
    """
    Mức vi phạm của TỪNG ràng buộc (đơn vị gốc, không bình phương).

    Inequality:
        g(x) >= 0   ->   violation = max(0, -g(x))

    Equality:
        h(x) = 0    ->   violation = |h(x)|
    """

    result = np.empty(len(constraints))

    for i, constraint in enumerate(constraints):

        value = constraint_value(i, x)

        # Không nằm trong miền xác định
        if not np.isfinite(value):
            result[i] = OUT_OF_DOMAIN_VIOLATION
            continue

        if constraint.kind == "ineq":
            result[i] = max(0.0, -value)

        else:
            result[i] = abs(value)

    return result


def total_constraint_violation(x, constraints, constraint_value):
    """
    Tổng mức vi phạm.

    Dùng tổng trị tuyệt đối (không bình phương) để con số báo cáo
    cùng đơn vị với tolerance - bình phương làm vi phạm 1e-6 hiện
    thành 1e-12, trông như đã khả thi trong khi thực ra thì chưa.
    """

    if not constraints:
        return 0.0

    return float(
        constraint_violations(
            x, constraints, constraint_value
        ).sum()
    )


def max_constraint_violation(x, constraints, constraint_value):
    """Vi phạm lớn nhất - đây mới là đại lượng đem so với tolerance."""

    if not constraints:
        return 0.0

    return float(
        constraint_violations(
            x, constraints, constraint_value
        ).max()
    )


def is_feasible(
    x,
    constraints,
    constraint_value,
    tolerance=FEASIBILITY_TOLERANCE
):

    return max_constraint_violation(
        x, constraints, constraint_value
    ) <= tolerance


def is_better(
    objective_a, violation_a,
    objective_b, violation_b,
    tolerance=FEASIBILITY_TOLERANCE
):
    """
    Quy tắc so sánh của Deb (feasibility rules):

        1. Nghiệm khả thi luôn tốt hơn nghiệm bất khả thi.
        2. Hai nghiệm cùng khả thi     -> so f(x).
        3. Hai nghiệm cùng bất khả thi -> so mức vi phạm.

    Trả về True nếu A tốt hơn B.
    """

    feasible_a = violation_a <= tolerance
    feasible_b = violation_b <= tolerance

    if feasible_a != feasible_b:
        return feasible_a

    if feasible_a:
        return objective_a < objective_b

    return violation_a < violation_b


# ============================================================
# 4. GENETIC ALGORITHM
# ============================================================

def genetic_algorithm(
    objective,
    constraints,
    constraint_value,
    init_box,

    population_size=100,
    generations=500,

    crossover_rate=0.9,
    mutation_rate=0.15,
    mutation_scale=0.08,

    elite_size=2,
    tournament_size=3,

    feasibility_tolerance=FEASIBILITY_TOLERANCE,
    epsilon_decay_fraction=0.7,
    epsilon_decay_power=4.0,

    reject_infeasible=True,
    rejection_min_share=0.2,

    seed=42,
):
    """
    GA mã hóa số thực cho bài toán tối ưu có ràng buộc.

    Xử lý ràng buộc bằng quy tắc khả thi của Deb kết hợp ngưỡng
    epsilon giảm dần (Takahama & Sato), KHÔNG dùng hệ số phạt tĩnh.

    Vì sao bỏ penalty tĩnh: với fitness = f + 1e6 * violation,
    thang phạt át hoàn toàn hàm mục tiêu, nên quần thể chỉ tối
    thiểu vi phạm rồi đứng yên tại một điểm bất kỳ trên mặt ràng
    buộc. Đo trên 'min x^2+y^2 s.t. x+y=1' (nghiệm đúng 0.5),
    penalty 1e6 cho trung bình 2.71 và xấu nhất 6.27 qua 8 seed.

    Ngưỡng epsilon nới lỏng ràng buộc ở giai đoạn đầu để quần thể
    còn di chuyển được dọc theo mặt ràng buộc - điều thiết yếu với
    ràng buộc đẳng thức, nơi tập khả thi có độ đo bằng 0 - rồi
    siết dần về feasibility_tolerance.
    """

    rng = np.random.default_rng(seed)

    init_box = np.asarray(init_box, dtype=float)

    lower = init_box[:, 0]
    upper = init_box[:, 1]

    variable_range = upper - lower

    n_variables = len(init_box)

    # --------------------------------------------------------
    # Initial population
    # --------------------------------------------------------

    population = rng.uniform(
        lower,
        upper,
        size=(population_size, n_variables)
    )

    # --------------------------------------------------------
    # Đánh giá: tách riêng mục tiêu và mức vi phạm
    # --------------------------------------------------------

    def evaluate(pop):

        objectives = np.empty(len(pop))

        # Vi phạm của TỪNG ràng buộc, cần cho việc loại cá thể bất khả thi
        tung_rang_buoc = np.zeros((len(pop), max(1, len(constraints))))

        for i, individual in enumerate(pop):

            objectives[i] = objective(individual)

            if constraints:
                tung_rang_buoc[i] = constraint_violations(
                    individual,
                    constraints,
                    constraint_value
                )

        violations = tung_rang_buoc.sum(axis=1) if constraints \
            else np.zeros(len(pop))

        # Điểm ngoài miền xác định: giữ hữu hạn để còn sắp xếp được
        objectives = np.where(
            np.isfinite(objectives),
            objectives,
            np.finfo(float).max
        )

        return objectives, violations, tung_rang_buoc

    # --------------------------------------------------------
    # Xếp hạng theo quy tắc Deb với ngưỡng epsilon
    # --------------------------------------------------------

    def rank_order(objectives, violations, epsilon):

        infeasible = violations > epsilon

        secondary = np.where(
            infeasible,
            violations,
            objectives
        )

        # lexsort: khóa cuối cùng là khóa chính
        return np.lexsort(
            (secondary, infeasible.astype(np.int64))
        )

    # --------------------------------------------------------
    # Lịch giảm epsilon
    # --------------------------------------------------------

    cutoff = max(
        1,
        int(epsilon_decay_fraction * generations)
    )

    def epsilon_at(generation, epsilon_0):

        if generation >= cutoff:
            return feasibility_tolerance

        factor = (
            1.0 - generation / cutoff
        ) ** epsilon_decay_power

        return max(
            feasibility_tolerance,
            epsilon_0 * factor
        )

    # --------------------------------------------------------
    # Tournament selection (theo thứ hạng Deb)
    # --------------------------------------------------------

    def tournament_selection(rank, be_lai_tao):

        indices = be_lai_tao[
            rng.integers(0, len(be_lai_tao), size=tournament_size)
        ]

        best_index = indices[
            np.argmin(rank[indices])
        ]

        return population[best_index].copy()

    # --------------------------------------------------------
    # Loại cá thể bất khả thi khỏi bể lai tạo
    # --------------------------------------------------------

    def be_lai_tao_cua(tung_rang_buoc):
        """
        Cá thể vi phạm ràng buộc thì không được làm cha mẹ.

        Chỉ áp cho những ràng buộc mà một phần đủ lớn của quần thể
        (rejection_min_share) đang thỏa mãn. Lý do: tập khả thi của
        ràng buộc ĐẲNG THỨC có độ đo bằng 0, gần như không cá thể nào
        thỏa ở những thế hệ đầu - loại thẳng thì cả quần thể chết và
        thuật toán không khởi động được. Ràng buộc kiểu 'r >= 0.1' thì
        luôn có sẵn nhiều cá thể thỏa, nên lọc được ngay từ đầu.
        """

        tat_ca = np.arange(len(tung_rang_buoc))

        if not constraints or not reject_infeasible:
            return tat_ca

        thoa = tung_rang_buoc <= feasibility_tolerance

        ap_dung = thoa.mean(axis=0) >= rejection_min_share

        if not ap_dung.any():
            return tat_ca

        giu = thoa[:, ap_dung].all(axis=1)

        # Còn quá ít cá thể thì không đủ đa dạng để lai tạo
        if giu.sum() < max(2 * elite_size, tournament_size):
            return tat_ca

        return tat_ca[giu]

    # --------------------------------------------------------
    # Blend crossover
    # --------------------------------------------------------

    def crossover(parent1, parent2):

        if rng.random() > crossover_rate:
            return (
                parent1.copy(),
                parent2.copy()
            )

        alpha = rng.uniform(
            -0.25,
            1.25,
            size=n_variables
        )

        child1 = (
            alpha * parent1
            + (1 - alpha) * parent2
        )

        child2 = (
            alpha * parent2
            + (1 - alpha) * parent1
        )

        # KHÔNG cắt về hộp: hộp chỉ dùng để khởi tạo và đặt thang
        # bước đột biến, không phải một cái lồng. Áp lực chọn lọc tự
        # kéo quần thể tới vùng tốt, kể cả khi vùng đó nằm ngoài hộp.
        # Đo trên 'min (x-10)^2+(y-10)^2' với hộp [-5,5]: có cắt thì
        # kẹt ở f=50 tại (5,5), bỏ cắt thì ra đúng f=0 tại (10,10).
        return child1, child2

    # --------------------------------------------------------
    # Gaussian mutation
    # --------------------------------------------------------

    def mutate(child):

        mutation_mask = (
            rng.random(n_variables)
            < mutation_rate
        )

        if np.any(mutation_mask):

            child[mutation_mask] += rng.normal(
                loc=0,
                scale=(
                    mutation_scale
                    * variable_range[mutation_mask]
                )
            )

        return child

    # --------------------------------------------------------
    # Evolution
    # --------------------------------------------------------

    history = []
    history_violation = []

    start_time = time.perf_counter()

    objectives, violations, tung_rang_buoc = evaluate(population)

    # Ngưỡng epsilon ban đầu: vi phạm trung vị của quần thể đầu tiên
    epsilon_0 = float(np.median(violations))

    best_index = rank_order(
        objectives, violations, feasibility_tolerance
    )[0]

    best_solution = population[best_index].copy()
    best_objective = objectives[best_index]
    best_violation = violations[best_index]

    def update_best():

        nonlocal best_solution, best_objective, best_violation

        for i in range(len(population)):
            if is_better(
                objectives[i], violations[i],
                best_objective, best_violation,
                feasibility_tolerance
            ):
                best_solution = population[i].copy()
                best_objective = objectives[i]
                best_violation = violations[i]

    for generation in range(generations):

        epsilon = epsilon_at(generation, epsilon_0)

        order = rank_order(objectives, violations, epsilon)

        rank = np.empty(population_size, dtype=np.int64)
        rank[order] = np.arange(population_size)

        be_lai_tao = be_lai_tao_cua(tung_rang_buoc)

        # Nghiệm tốt nhất từng gặp, xét theo tolerance thật
        update_best()

        history.append(best_objective)
        history_violation.append(best_violation)

        # Elitism
        new_population = [
            population[i].copy()
            for i in order[:elite_size]
        ]

        # Sinh thế hệ tiếp theo
        while len(new_population) < population_size:

            parent1 = tournament_selection(rank, be_lai_tao)
            parent2 = tournament_selection(rank, be_lai_tao)

            child1, child2 = crossover(
                parent1,
                parent2
            )

            new_population.append(
                mutate(child1)
            )

            if len(new_population) < population_size:
                new_population.append(
                    mutate(child2)
                )

        population = np.asarray(new_population)

        objectives, violations, tung_rang_buoc = evaluate(population)

    # --------------------------------------------------------
    # Final result
    # --------------------------------------------------------

    update_best()

    elapsed_time = time.perf_counter() - start_time

    # Thế hệ đầu tiên mà nghiệm tốt nhất từng gặp đã khả thi.
    # None nghĩa là chạy hết số thế hệ vẫn chưa thỏa mãn ràng buộc.
    feasible_at = next(
        (
            g
            for g, violation in enumerate(history_violation)
            if violation <= feasibility_tolerance
        ),
        None,
    )

    return build_result(
        best_solution,
        objective,
        constraints,
        constraint_value,
        elapsed_time,
        history=history,
        history_violation=history_violation,
        generations=generations,
        feasible_at=feasible_at,
        seed=seed,
    )


# ============================================================
# 5. SCIPY - SLSQP
# ============================================================

def build_result(
    x,
    objective,
    constraints,
    constraint_value,
    elapsed_time,
    **extra
):
    """Gói kết quả theo một định dạng chung cho mọi phương pháp."""

    x = np.asarray(x, dtype=float)

    result = {
        "x": x,

        "fun": objective(x),

        "feasible": is_feasible(
            x, constraints, constraint_value
        ),

        "violation": total_constraint_violation(
            x, constraints, constraint_value
        ),

        "max_violation": max_constraint_violation(
            x, constraints, constraint_value
        ),

        "time": elapsed_time,
    }

    result.update(extra)

    return result


def _run_slsqp(
    objective,
    constraints,
    constraint_value,
    x0
):
    """Một lần chạy SLSQP từ điểm khởi tạo x0."""

    scipy_constraints = [
        {
            # SLSQP: "ineq" nghĩa là g(x) >= 0, "eq" nghĩa là
            # h(x) = 0 - trùng đúng dạng đã chuẩn hóa ở
            # parse_constraint, nên dùng thẳng constraint.kind
            "type": constraint.kind,

            "fun":
                lambda x, i=i:
                constraint_value(i, x)
        }
        for i, constraint in enumerate(constraints)
    ]

    return minimize(
        objective,

        x0=x0,

        method="SLSQP",

        constraints=scipy_constraints,

        options={
            "maxiter": 2000,
            "ftol": 1e-12,
            "disp": False,
        }
    )


def scipy_slsqp(
    objective,
    constraints,
    constraint_value,
    init_box,
    n_starts=30,
    seed=42,
):
    """
    SLSQP đa điểm khởi tạo (multi-start).

    Vì sao cần nhiều điểm: SLSQP là thuật toán CỤC BỘ, nó hội tụ về
    điểm dừng KKT gần nhất chứ không phải cực tiểu toàn cục. Với
    'min x*y s.t. x^2+y^2=1' (nghiệm đúng -0.5), khởi tạo từ trung
    điểm init_box (0,0) cho ra +0.5 - tức là điểm CỰC ĐẠI.

    SLSQP KHÔNG nhận init_box: chặn trên/dưới nếu cần thì viết thành
    ràng buộc (ví dụ 'x >= 0'), để GA và SLSQP giải đúng cùng một
    bài toán. init_box chỉ dùng để rải điểm khởi tạo.
    """

    init_box = [tuple(b) for b in init_box]

    rng = np.random.default_rng(seed)

    lower = np.array([b[0] for b in init_box], dtype=float)
    upper = np.array([b[1] for b in init_box], dtype=float)

    start_points = [(lower + upper) / 2]

    if n_starts > 1:
        start_points.extend(
            rng.uniform(
                lower, upper,
                size=(n_starts - 1, len(init_box))
            )
        )

    start_time = time.perf_counter()

    best = None
    n_success = 0

    for x0 in start_points:

        try:
            raw = _run_slsqp(
                objective,
                constraints,
                constraint_value,
                x0
            )

        except Exception:
            continue

        x = np.asarray(raw.x, dtype=float)

        value = objective(x)

        if not np.isfinite(value):
            continue

        violation = total_constraint_violation(
            x, constraints, constraint_value
        )

        n_success += bool(raw.success)

        if best is None or is_better(
            value, violation,
            best[1], best[2]
        ):
            best = (x, value, violation, raw)

    elapsed_time = time.perf_counter() - start_time

    if best is None:
        raise RuntimeError(
            "SLSQP không tìm được nghiệm hữu hạn từ bất kỳ "
            "điểm khởi tạo nào."
        )

    x, _, _, raw = best

    return build_result(
        x,
        objective,
        constraints,
        constraint_value,
        elapsed_time,
        success=bool(raw.success),
        message=str(raw.message),
        iterations=int(raw.nit),
        n_starts=len(start_points),
        n_success=n_success,
    )


print(f"numpy {np.__version__} | scipy {scipy.__version__} | sympy {sympy.__version__}")
print("Tên dành riêng (không dùng làm biến):", ", ".join(sorted(RESERVED_NAMES)))


# ------------------------------------------------------------------
# Hiển thị dạng ký hiệu toán học (LaTeX)
# ------------------------------------------------------------------
from IPython.display import Markdown, display


def _num(value, digits=10):
    """Số dạng LaTeX; chuyển sang ký hiệu khoa học khi quá lớn hoặc quá nhỏ."""
    if not np.isfinite(value):
        return r"\infty" if value > 0 else r"-\infty"
    if value != 0 and (abs(value) >= 1e6 or abs(value) < 1e-4):
        mantissa, exponent = f"{value:.4e}".split("e")
        return mantissa + r" \times 10^{" + str(int(exponent)) + "}"
    return f"{value:.{digits}f}"


def show_problem(objective_expr, constraints, variables):
    """Phát biểu bài toán tối ưu dưới dạng toán học chuẩn."""
    bien = ", ".join(sympy.latex(v) for v in variables)
    dong = [r"&\underset{" + bien + r"}{\text{minimize}} \quad && f\left("
            + bien + r"\right) = " + sympy.latex(objective_expr) + r" \\"]

    dau = True
    for c in constraints:
        quan_he = r"\ \ge\ 0" if c.kind == "ineq" else r"\ =\ 0"
        nhan = r"\text{subject to}" if dau else ""
        dong.append("&" + nhan + r" \quad && " + sympy.latex(c.expr) + quan_he + r" \\")
        dau = False

    display(Markdown("$$\n\\begin{aligned}\n" + "\n".join(dong) + "\n\\end{aligned}\n$$"))


def show_solution(name, result, variables):
    """Nghiệm tìm được và thời gian chạy."""
    toado = r" \\ ".join(
        sympy.latex(v) + " &= " + _num(x) for v, x in zip(variables, result["x"])
    )

    khoi = [
        "**" + name + "**",
        "",
        r"$$\begin{aligned}" + toado + r"\end{aligned}$$",
        r"$$f^{*} = " + _num(result["fun"]) + r"$$",
        "",
        "| | |",
        "|---|---|",
        "| Thời gian chạy | $" + _num(result["time"], 6) + r"\ \text{s}$ |",
    ]

    if "feasible_at" in result:
        if result["feasible_at"] is None:
            khoi.append("| Thỏa mãn ràng buộc | chưa đạt sau $"
                        + str(result["generations"]) + "$ thế hệ |")
        else:
            khoi.append("| Thỏa mãn ràng buộc từ thế hệ | $"
                        + str(result["feasible_at"]) + " / "
                        + str(result["generations"]) + "$ |")

    if "success" in result:
        khoi.append("| SLSQP hội tụ | $" + str(result["n_success"]) + "/"
                    + str(result["n_starts"]) + "$ điểm khởi tạo |")

    # Chỉ hiện khi nghiệm KHÔNG khả thi. Lúc bình thường không có dòng
    # này; nhưng nếu thuật toán thất bại thì phải báo, không thì người
    # dùng đọc một con số vô nghĩa mà tưởng là kết quả.
    if not result["feasible"]:
        khoi += [
            "",
            "> ⚠️ **Nghiệm này KHÔNG thỏa mãn ràng buộc** — vi phạm $"
            + _num(result["max_violation"])
            + r"$, vượt dung sai $10^{-6}$. Giá trị $f^{*}$ ở trên không dùng được.",
        ]

    display(Markdown("\n".join(khoi)))


def show_statistics(runs):
    """Thống kê qua nhiều lần chạy GA độc lập."""
    gia_tri = np.array([r["fun"] for r in runs if np.isfinite(r["fun"])])

    if len(gia_tri) == 0:
        display(Markdown("*Không lần chạy nào cho giá trị hữu hạn.*"))
        return

    display(Markdown("\n".join([
        "**Thống kê GA qua " + str(len(runs)) + " lần chạy độc lập**",
        "",
        "| | |",
        "|---|---|",
        r"| Tốt nhất | $\min f = " + _num(gia_tri.min()) + "$ |",
        r"| Trung bình | $\bar{f} = " + _num(gia_tri.mean()) + "$ |",
        r"| Tệ nhất | $\max f = " + _num(gia_tri.max()) + "$ |",
        r"| Độ lệch chuẩn | $\sigma = " + _num(gia_tri.std()) + "$ |",
    ])))


def show_comparison(ga_result, scipy_result, variables):
    """Bảng so sánh: chỉ f*, thời gian và các biến."""
    cot_bien = " | ".join("$" + sympy.latex(v) + "$" for v in variables)

    dong = [
        r"| Phương pháp | $f^{*}$ | Thời gian (s) | " + cot_bien + " |",
        "|---|---|---|" + "---|" * len(variables),
    ]

    for ten, r in (("Genetic Algorithm", ga_result), ("SciPy SLSQP", scipy_result)):
        toado = " | ".join("$" + _num(x) + "$" for x in r["x"])
        dong.append("| " + ten + " | $" + _num(r["fun"]) + "$ | $"
                    + _num(r["time"], 6) + "$ | " + toado + " |")

    display(Markdown("\n".join(dong)))


numpy 1.26.4 | scipy 1.17.1 | sympy 1.14.0
Tên dành riêng (không dùng làm biến): Abs, E, abs, acos, asin, atan, cos, cosh, e, exp, ln, log, pi, sin, sinh, sqrt, tan, tanh


---
## ① Nhập bài toán

Điền vào form rồi bấm **Áp dụng bài toán**.

**Ràng buộc** — mỗi dòng một cái, để trống nếu không có:

| Gõ | Ý nghĩa |
|---|---|
| `x <= 2` &nbsp; `x >= 0` &nbsp; `x + y = 1` | một ràng buộc |
| `0 <= x <= 2` | chuỗi kép — tự tách thành `0 <= x` và `x <= 2` |
| `x < 2` | ❌ không nhận — tối ưu số cần miền đóng, dùng `<=` |

Không có miền tìm kiếm riêng: muốn chặn biến thì viết thành ràng buộc.

**Hàm mục tiêu** — `x^2` hoặc `x**2`, `2x` = `2*x`, hàm `sqrt exp log ln abs sin cos tan`.

⚠️ **Tích hai biến phải có dấu `*`**: gõ `2xy` thì `xy` thành một biến mới chứ
không phải `2*x*y`. Viết `2*x*y`. Chương trình sẽ báo lỗi nếu phát hiện.

In [7]:
import ipywidgets as W
from IPython.display import clear_output, display

# ------------------------------------------------------------------
# Giá trị đặt cứng, không hỏi trong form
# ------------------------------------------------------------------

# Số điểm khởi tạo của SLSQP. SLSQP là thuật toán CỤC BỘ nên chỉ tìm
# được điểm dừng KKT gần điểm xuất phát nhất. Chạy từ một điểm duy
# nhất là không đủ: với 'min x*y s.t. x^2+y^2=1' (nghiệm đúng -0.5),
# xuất phát từ (0,0) cho ra +0.5 - tức CỰC ĐẠI.
SLSQP_N_STARTS = 30

_LBL = {"description_width": "130px"}
_WIDE = W.Layout(width="580px")

w_objective = W.Text(
    value="x^2 + y^2",
    description="Hàm mục tiêu",
    placeholder="ví dụ:  x^2 + y^2",
    layout=_WIDE, style=_LBL, continuous_update=False,
)

w_constraints = W.Textarea(
    value="x + y = 1",
    description="Ràng buộc",
    placeholder="mỗi dòng một ràng buộc — để trống nếu không có",
    layout=W.Layout(width="580px", height="80px"), style=_LBL,
    continuous_update=False,
)

# Ba ô tham số nằm chung một hàng.
#
# KHÔNG dùng tham số `description` của widget: bề ngang của nhãn đó
# nằm NGOÀI layout.width, nên đặt width bao nhiêu cũng vẫn tràn và
# sinh thanh cuộn ngang. Dùng Label riêng thì bề ngang tính được
# chính xác: (90 + 80) x 3 = 510px, thừa chỗ trong khung 580px.
def _o_tham_so(nhan, gia_tri):
    o = W.IntText(value=gia_tri, layout=W.Layout(width="80px"))
    hop = W.HBox(
        [W.Label(nhan, layout=W.Layout(width="90px")), o],
        layout=W.Layout(width="170px"),
    )
    return o, hop


w_population, _hop_population = _o_tham_so("Quần thể", 100)
w_generations, _hop_generations = _o_tham_so("Số thế hệ", 500)
w_runs, _hop_runs = _o_tham_so("Số lần chạy", 5)

status = W.Output()
apply_button = W.Button(description="Áp dụng bài toán", button_style="primary",
                        icon="check", layout=W.Layout(width="200px"))

problem_ready = False


def _read_form():
    """Đọc form và dựng bài toán. Ném ValueError nếu cú pháp sai."""
    objective_text = w_objective.value.strip()
    constraint_texts = [line.strip() for line in w_constraints.value.splitlines()
                        if line.strip()]
    return build_problem(objective_text, constraint_texts), objective_text, constraint_texts


def _on_change(_=None):
    """Gõ xong hàm mục tiêu / ràng buộc thì cập nhật danh sách biến ngay."""
    with status:
        clear_output()
        try:
            (expr, _cons, found, *_), _, _ = _read_form()
        except Exception as error:
            print("✗", error)
            return
        print("Đọc được :", expr)
        print("Biến     :", ", ".join(str(v) for v in found))
        print("\nBấm 'Áp dụng bài toán' để chạy.")


def _apply(_=None):
    global OBJECTIVE, CONSTRAINTS, objective_expr, constraints, variables
    global objective, constraint_value, constraint_gradient, init_box
    global POPULATION_SIZE, GENERATIONS, N_RUNS, problem_ready

    with status:
        clear_output()
        problem_ready = False
        try:
            parsed, OBJECTIVE, CONSTRAINTS = _read_form()
        except Exception as error:
            print("✗", error)
            return

        (objective_expr, constraints, variables,
         objective, constraint_value, constraint_gradient) = parsed

        # Hộp chỉ dùng để sinh quần thể ban đầu và đặt thang bước đột
        # biến. GA không bị cắt về hộp nên vẫn đi ra ngoài được, và
        # SLSQP không nhận bounds - cả hai giải đúng cùng một bài toán.
        # Muốn chặn thật thì viết thành ràng buộc, ví dụ 'x >= 0'.
        init_box = [INIT_BOX] * len(variables)

        # Chuẩn hóa thang đo ràng buộc: mức vi phạm không phụ thuộc cách viết
        constraint_value = build_scaled_constraint_value(
            constraints, constraint_value, constraint_gradient, init_box
        )

        POPULATION_SIZE = int(w_population.value)
        GENERATIONS = int(w_generations.value)
        N_RUNS = int(w_runs.value)
        problem_ready = True

        show_problem(objective_expr, constraints, variables)


w_objective.observe(_on_change, names="value")
w_constraints.observe(_on_change, names="value")
apply_button.on_click(_apply)

display(W.VBox([
    W.HTML("<b>Bài toán</b>"),
    w_objective,
    w_constraints,
    W.HTML("<b>Tham số GA</b>"),
    W.HBox([_hop_population, _hop_generations, _hop_runs],
           layout=W.Layout(width="580px")),
    apply_button,
    status,
]))

# Áp dụng luôn với giá trị đang có trong form, để "Run All" chạy được ngay.
# Sau khi sửa form thì bấm nút "Áp dụng bài toán" rồi chạy lại từ cell ②.
_apply()

---
## ② Chạy Genetic Algorithm

In [8]:
assert problem_ready, (
    "Bài toán ở cell ① chưa hợp lệ. Xem thông báo lỗi ngay dưới form ở cell ①, "
    "sửa lại rồi bấm nút 'Áp dụng bài toán'."
)

ga_runs = [
    genetic_algorithm(
        objective,
        constraints,
        constraint_value,
        init_box,
        population_size=POPULATION_SIZE,
        generations=GENERATIONS,
        seed=42 + i,
    )
    for i in range(N_RUNS)
]

# Chọn lần chạy tốt nhất theo quy tắc khả thi của Deb
ga_result = functools.reduce(
    lambda a, b: b if is_better(
        b["fun"], b["violation"], a["fun"], a["violation"]
    ) else a,
    ga_runs,
)

show_solution(
    "GENETIC ALGORITHM — lần chạy tốt nhất",
    ga_result, variables,
)

if N_RUNS > 1:
    show_statistics(ga_runs)

**GENETIC ALGORITHM — lần chạy tốt nhất**

$$\begin{aligned}x &= 0.4977131810 \\ y &= 0.5022854048\end{aligned}$$
$$f^{*} = 0.5000090384$$

| | |
|---|---|
| Thời gian chạy | $9.583584\ \text{s}$ |
| Thỏa mãn ràng buộc từ thế hệ | $355 / 500$ |

**Thống kê GA qua 5 lần chạy độc lập**

| | |
|---|---|
| Tốt nhất | $\min f = 0.5000090384$ |
| Trung bình | $\bar{f} = 0.5006003028$ |
| Tệ nhất | $\max f = 0.5014113080$ |
| Độ lệch chuẩn | $\sigma = 0.0006490879$ |

---
## ③ Chạy SciPy SLSQP (package có sẵn, dùng làm mốc so sánh)

In [9]:
scipy_result = scipy_slsqp(
    objective,
    constraints,
    constraint_value,
    init_box,
    n_starts=SLSQP_N_STARTS,
    seed=42,
)

show_solution(
    "SCIPY SLSQP — đa điểm khởi tạo",
    scipy_result, variables,
)

**SCIPY SLSQP — đa điểm khởi tạo**

$$\begin{aligned}x &= 0.4999999992 \\ y &= 0.5000000008\end{aligned}$$
$$f^{*} = 0.5000000000$$

| | |
|---|---|
| Thời gian chạy | $0.149458\ \text{s}$ |
| SLSQP hội tụ | $30/30$ điểm khởi tạo |

---
## ④ Bảng so sánh

In [10]:
show_comparison(ga_result, scipy_result, variables)

| Phương pháp | $f^{*}$ | Thời gian (s) | $x$ | $y$ |
|---|---|---|---|---|
| Genetic Algorithm | $0.5000090384$ | $9.583584$ | $0.4977131810$ | $0.5022854048$ |
| SciPy SLSQP | $0.5000000000$ | $0.149458$ | $0.4999999992$ | $0.5000000008$ |